In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [2]:
#!pip install langchain_text_splitters langchain_community

In [3]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

print("✅ Environment ready!")

✅ Environment ready!


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

text_path="/content/drive/MyDrive/Datasets/sample_data/notes.txt"

if Path(text_path).exists():
  # Load the document
  loader = TextLoader(text_path)
  documents = loader.load()

  print(f"📄 Original document:{len(documents[0].page_content)} characters\n")

  # Create Splitter
  splitter = RecursiveCharacterTextSplitter(
      chunk_size = 200,
      chunk_overlap = 20,
      length_function = len,
      separators = [". "]
  )
  # Split the document
  chunks = splitter.split_documents(documents)

  print(f"✂️ Split into {len(chunks)} chunks\n")

  #Examine first 3 chunks
  for i, chunk in enumerate(chunks[:3], 1):
    print(f"{'='*70}")
    print(f"Chunk {i} ({len(chunk.page_content)} chars):")
    print(f"{'='*70}")
    print(chunk.page_content[:300]+'...' if len(chunk.page_content) > 300 else chunk.page_content)
    print()
else:
  print(f"❌ File not found: {text_path}")

📄 Original document:8567 characters

✂️ Split into 13 chunks

Chunk 1 (200 chars):
LANGCHAIN STUDY NOTES - RAG IMPLEMENTATION

Date: January 15, 2025
Topic: Retrieval-Augmented Generation with LangChain 1.0+


CORE CONCEPTS
-------------

1

Chunk 2 (192 chars):
. Document Object Structure
   - page_content: The actual text content
   - metadata: Dictionary with additional information (source, page, date, etc.)
   - id: Unique identifier (optional)

2

Chunk 3 (236 chars):
. LCEL (LangChain Expression Language)
   - Uses pipe operator | to chain components
   - More readable than nested function calls
   - Better error handling and debugging
   - Example: retriever | format_docs | prompt | llm | parser

3



In [6]:
# Chunk overlap
if Path(text_path).exists():
  docs = TextLoader(text_path).load()

  # Splitter with overlap
  splitter_with_overlap = RecursiveCharacterTextSplitter(
      chunk_size=500,
      chunk_overlap=100
  )

  chunks = splitter_with_overlap.split_documents(docs)
  print("🔍 Examining overlap between chunks:\n")

  # Show overlap between chunk1 and chunk2
  if len(chunks)>=2:
    chunk1_end = chunks[0].page_content[-150:]
    chunk2_start = chunks[1].page_content[:150]

    print("Chunk 1 ending:")
    print(f"  ...{chunk1_end}")
    print("chunk 2 beginning:")
    print(f"  {chunk2_start}...")
    print("\n💡 Notice the overlap? This preserves the context!")

🔍 Examining overlap between chunks:

Chunk 1 ending:
  ...ontent: The actual text content
   - metadata: Dictionary with additional information (source, page, date, etc.)
   - id: Unique identifier (optional)
chunk 2 beginning:
  2. LCEL (LangChain Expression Language)
   - Uses pipe operator | to chain components
   - More readable than nested function calls
   - Better error ...

💡 Notice the overlap? This preserves the context!


## Custom separators for code

In [8]:
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter

# Python code example

python_code = '''
def calculate_total(items):
    """Calculate total price of items."""
    total = 0
    for item in items:
        total += item['price']
    return total

def apply_discount(total, discount_percent):
    """Apply discount to total."""
    discount = total * (discount_percent / 100)
    return total - discount

class ShoppingCart:
    def __init__(self):
        self.items = []

    def add_item(self, item):
        self.items.append(item)
'''

python_splitter = RecursiveCharacterTextSplitter.from_language(
    language = Language.PYTHON,
    chunk_size=200,
    chunk_overlap=40
)

code_chunks = python_splitter.split_text(python_code)

print(f"✂️ split code into {len(code_chunks)} chunks:\n")
for i, chunk in enumerate(code_chunks, 1):
  print(f"Chunk {i}")
  print(chunk)
  print("-"*50)

✂️ split code into 3 chunks:

Chunk 1
def calculate_total(items):
    """Calculate total price of items."""
    total = 0
    for item in items:
        total += item['price']
    return total
--------------------------------------------------
Chunk 2
def apply_discount(total, discount_percent):
    """Apply discount to total."""
    discount = total * (discount_percent / 100)
    return total - discount
--------------------------------------------------
Chunk 3
class ShoppingCart:
    def __init__(self):
        self.items = []
    
    def add_item(self, item):
        self.items.append(item)
--------------------------------------------------


# Character TextSplitter

In [9]:
from langchain_text_splitters import CharacterTextSplitter

# Sample text with clear paragraph breaks
sample_text = """First paragraph about machine learning.
It has multiple sentences. This is important context.

Second paragraph about deep learning.
Neural networks are powerful. They learn from data.

Third paragraph about transformers.
Attention mechanisms are key. They revolutionized NLP.
"""

simple_splitter = CharacterTextSplitter(
    separator = "\n\n",
    chunk_size=300,
    chunk_overlap=20
)

chunks = simple_splitter.split_text(sample_text)

print(f"Split into {len(chunks)} chunks:\n")
for i, chunk in enumerate(chunks, 1):
  print(f"Chunk {i}: {chunk.strip()}\n")

Split into 1 chunks:

Chunk 1: First paragraph about machine learning.
It has multiple sentences. This is important context.

Second paragraph about deep learning.
Neural networks are powerful. They learn from data.

Third paragraph about transformers.
Attention mechanisms are key. They revolutionized NLP.



In [10]:
from langchain_text_splitters import HTMLHeaderTextSplitter

# Load the HTML blog post
html_path = "/content/drive/MyDrive/Datasets/sample_data/blog_post.html"

if Path(html_path).exists():
  # Read HTML content
  with open(html_path, 'r', encoding='utf-8') as f:
    html_content=f.read()

  # Define headers to split on
  headers_to_split_on = [
      ("h1", "Title"),
      ("h2", "Section"),
      ("h3", "Subsection")
  ]

  # Create splitter
  html_splitter = HTMLHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

  # Split the HTML
  html_chunks = html_splitter.split_text(html_content)

  print(f"✂️ Split HTML into {len(html_chunks)} sections\n")

  # Show first 3 chunks with metadata
  for i, chunk in enumerate(html_chunks[:6], 1):
    print(f"{'='*70}")
    print(f"Content (first 200 chars):{chunk.page_content[:400]}...")
    print()
else:
  print(f"❌ HTML file not found:{html_path}")

✂️ Split HTML into 48 sections

Content (first 200 chars):Building Intelligent Applications with RAG...

Content (first 200 chars):| |  
By Dr. Amanda Foster  
January 15, 2025  
12 min read...

Content (first 200 chars):Introduction...

Content (first 200 chars):In the rapidly evolving landscape of artificial intelligence, Retrieval-Augmented Generation (RAG) has emerged as a game-changing approach for building intelligent applications. Unlike traditional chatbots that rely solely on the knowledge embedded in their training data, RAG systems combine the power of information retrieval with language generation to produce more accurate, contextual, and up-to...

Content (first 200 chars):What is Retrieval-Augmented Generation?...

Content (first 200 chars):RAG is an architectural pattern that enhances language model outputs by incorporating relevant information retrieved from external knowledge sources. The process typically involves three key steps:  
When a user asks a question, the sy

# Recursive JSON Splitter

In [11]:
from langchain_text_splitters import RecursiveJsonSplitter
import json

# Load json data
json_path = "/content/drive/MyDrive/Datasets/sample_data/api_response.json"

if Path(json_path).exists():
  with open(json_path, 'r') as f:
    json_data = json.load(f)

  # Create splitter
  json_splitter = RecursiveJsonSplitter(
      max_chunk_size=200,
      min_chunk_size=20
  )

  # Split
  json_chunks = json_splitter.split_text(
      json_data = json_data,
      convert_lists = True
  )

  print(f"✂️ Split JSON into {len(json_chunks)} chunks")

  # Show first chunk
  print("First chunk:")
  print(json.dumps(json_chunks[0], indent=2)[:500]+"...")
else:
  print(f"❌ JSON file not found:{json_path}")

✂️ Split JSON into 27 chunks
First chunk:
"{\"api_version\": \"v2.0\", \"timestamp\": \"2025-01-15T10:30:00Z\", \"total_results\": 5}"...


# Token Text Splitter

In [12]:
from langchain_text_splitters import TokenTextSplitter

# Sample text
text = """The transformer architecture, introduced in the paper 'Attention Is All You Need',
revolutionized natural language processing. It uses self-attention mechanisms to process
sequences in parallel, making it much faster than recurrent neural networks."""

# Token-based splitter
token_splitter = TokenTextSplitter(
    chunk_size=10,
    chunk_overlap=5,
    encoding_name="cl100k_base"
)

token_chunks = token_splitter.split_text(text)

print(f"Split into {len(token_chunks)} token-based chunks:\n")
for i, chunk in enumerate(token_chunks, 1):
  print(f"Chunk {i}:{chunk}\n")

Split into 8 token-based chunks:

Chunk 1:The transformer architecture, introduced in the paper 'Attention

Chunk 2: in the paper 'Attention Is All You Need',

Chunk 3: Is All You Need', 
revolutionized natural language

Chunk 4: 
revolutionized natural language processing. It uses self

Chunk 5: processing. It uses self-attention mechanisms to process

Chunk 6:-attention mechanisms to process 
sequences in parallel,

Chunk 7: 
sequences in parallel, making it much faster than

Chunk 8: making it much faster than recurrent neural networks.



# Chunk Size and Overlap Optimization

In [14]:
# compare different chunk sizes
if Path(text_path).exists():
  docs = TextLoader(text_path).load()

  chunk_sizes=[500, 1000, 1500, 2000]

  print("📊 Chunk Size comparision:\n")
  print(f"{'Size':<8} {'Chunks':<10} {'Avg Length':<12} {'Overlap %'}")
  print("-"*50)

  for size in chunk_sizes:
    overlap = int(size*0.2)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=overlap
    )

    chunks = splitter.split_documents(docs)
    avg_length = sum(len(c.page_content) for c in chunks)/len(chunks)
    overlap_pct = (overlap/size)*100

    print(f"{size:<8} {len(chunks):<10} {avg_length:<12.0f} {overlap_pct:.0f}%")

📊 Chunk Size comparision:

Size     Chunks     Avg Length   Overlap %
--------------------------------------------------
500      25         356          20%
1000     12         819          20%
1500     8          1229         20%
2000     6          1650         20%
